# 06 — Integral windup

**The integrator does not know the engine has run out.**

Notebooks 03, 04 and 05 ran with the torque ceiling lifted — *assume for now an engine that
delivers whatever we ask*. Every gain in them is honest under that assumption, and the
assumption is what made the linear story tellable: a proportional term that scales, an
integrator that converges on the holding torque, a derivative that brakes on approach.

This notebook removes it. The engine has 150 N.m and not one more, and the loop from notebook
05 is about to ask for more than that for a sustained stretch — not as a transient spike during
a step, but for as long as the hill lasts.

The failure that follows is not a gain that was set too high. It is a controller carrying out
its own rule correctly while the plant it is attached to stops listening.

## Setup

Same bootstrap as every notebook in this set. The ceiling goes back where it belongs.

In [ ]:
isdefined(Main, :Lecture01Support) || include("support.jl")

# Bound by hand rather than with `using`. If this kernel ever ran an earlier `include` of
# support.jl there are two copies of the module, every name both export is ambiguous, and no
# amount of re-running `using` resolves it. An explicit binding does, so this cell repairs a
# kernel in that state instead of needing it restarted. Editing support.jl still needs a
# restart, because the guard above deliberately will not load a second copy.
for n in names(Lecture01Support)
    n === :Lecture01Support && continue
    Core.eval(Main, :($n = Lecture01Support.$n))
end

In [ ]:
setup()

## 1. Actuators are not linear

Every block diagram so far has been a lie by omission. `u = k·e` says that doubling the error
doubles the command, and it keeps saying it at 300 N.m, at 1200 N.m, at 17 800 N.m — the
proportional term in notebook 03's `k = 890` run. Nothing in those equations knows the engine
tops out at 150.

Real actuators saturate. A valve is fully open, a rudder is hard over, an amplifier is at the
rail, an engine is at wide-open throttle. Past that point the actuator has one behavior: it
delivers its limit and ignores the rest of the command.

Two things are worth separating before we look at a plot.

- **Saturation on its own is not a fault.** A loop that asks for more than it can have, gets
  the limit, and approaches its setpoint more slowly than the linear design promised, has
  behaved sensibly. Notebook 08 ends on exactly that: three heuristic tunings, all commanding
  several times the engine, all still arriving.
- **What breaks is the integrator's bookkeeping.** The integral term's whole rule is *keep
  accumulating until the error is zero*. It reads the error. It does not read the delivered
  torque. So while the engine is pinned at its limit and the error refuses to close, the rule
  says the same thing it always says — accumulate — and the integrator climbs to a value that
  no longer corresponds to any torque the engine can produce.

The gap between what was commanded and what was delivered is the whole subject of this
notebook, and it opens the moment the command crosses 150 N.m.

### The model this runs

Two pieces already in the library, plus one that is not.

`Lecture1.CruiseLoop` is unchanged from notebooks 03 to 05 — and it has always had a `grade`
input, tied to a constant zero by the flat-road step harness. This notebook is the first to
drive it. `Vehicle.GradeProfile` supplies the road: flat, then a climb of a given gradient
starting at `start_time`, and — with a finite `duration` — flat again afterwards. That second
flat stretch is where the damage shows, so the `duration` parameter exists for this notebook.

Sources: [`dyad/Lecture1/CruiseLoop.dyad`](../../dyad/Lecture1/CruiseLoop.dyad),
[`dyad/Vehicle/GradeProfile.dyad`](../../dyad/Vehicle/GradeProfile.dyad) and the force it
applies in [`dyad/Vehicle/GradeForce.dyad`](../../dyad/Vehicle/GradeForce.dyad).

What does not exist yet is the harness that connects the two, and the clamping controller of
section 4. Both are named where the cells that need them would go.

In [ ]:
show_dyad("GradeProfile")

## 2. The climb

The scenario is a car cruising at 130 km/h on the flat that meets a sustained 10% climb at
t = 30 s, and is still climbing long enough for the loop to settle into the saturated state.

The arithmetic is notebook 01's force balance with one term added, and it says the climb cannot
be held before anything is simulated:

| | |
|---|---|
| gravity along a 10% slope, `m·g·sin(atan 0.10)` | 1366 N |
| aerodynamic drag at 130 km/h | 493 N |
| rolling resistance, normal load shed by `cos α` | 164 N |
| **needed to hold 130 km/h uphill** | **2023 N** |
| **available, `T_max·i/r`** | **1935 N** |

The car is 88 N short. It cannot hold 130 km/h on that hill at any gain, with any controller,
and it settles instead at about 118 km/h — the speed at which the drag it sheds by slowing down
makes up the difference. That is the correct behavior of a car with a finite engine on a hill
that is too steep for it, and a lecture should say so before the plot arrives, so that nobody
reads the speed drop itself as the failure.

The failure is elsewhere. Watch the integrator while the speed is stuck 12 km/h below its
setpoint.

> **Cell not written yet — the climb, with no anti-windup.**
>
> Blocked on the climb harness: a scenario tying `CruiseLoop`'s `grade` input to `GradeProfile`,
> with an analysis that exposes the gradient, its start time and its duration. It will plot
> speed, commanded torque, delivered torque and the integrator state on one time axis. Nothing
> is stubbed here on purpose: a plot of invented data would look exactly like a result.

## 3. Why the overshoot is coming

Nothing has gone wrong yet. The car is slower than asked, the engine is flat out, and a
passenger would report that the car is climbing a steep hill. The controller, meanwhile, is
accumulating.

Follow it in four steps, the same way notebook 04 followed the overshoot on a step.

1. **The error is stuck open.** About 12 km/h, and it will not close, because closing it needs
   torque that does not exist. The integrator's stopping condition is never met.
2. **So the sum grows, without limit and without effect.** Every second of a sustained error
   adds to it. The commanded torque climbs past 150, past 300, past whatever the hill and the
   gain between them produce — and the delivered torque does not move, because it has been at
   the limit since early in the climb. The command and the delivery have come apart, and
   nothing in the loop is measuring the gap.
3. **The road flattens, and the car starts accelerating.** The load that justified all that
   accumulation is gone in an instant. The error closes fast.
4. **But the command has to come back down before anything changes.** Reducing a command of
   400 N.m to 300 N.m changes the delivered torque by exactly nothing: both saturate to 150.
   The engine stays at full torque through the entire descent from 400 to 150, and the only
   thing that brings the sum down is *negative* error — the car being faster than its setpoint.

Step 4 is the one to sit with. The car must overshoot, and keep overshooting, for as long as it
takes to unwind everything the hill put into the integrator. The overshoot is not a symptom of
the windup. The overshoot **is** the unwinding, and its size is set by how long the hill lasted
rather than by any gain in the controller.

This is why windup is worse than it first looks. A tuning that is impeccable on a flat-road step
carries a fault that only appears after a long saturated stretch, and the longer the stretch the
worse the excursion — a controller whose error grows with the duration of the disturbance.

> **Cell not written yet — the overshoot after the climb.**
>
> Blocked on the same harness run with a finite `duration`, far enough past the end of the climb
> to capture the whole excursion, with the peak annotated. Nothing is stubbed here on purpose: a
> plot of invented data would look exactly like a result.

## 4. Clamping

The fix is to notice that the actuator has stopped listening, and stop integrating while it
has. Two conditions, both required:

1. **The output is saturating** — the command is at or past the limit.
2. **The error has the same sign as the output** — integrating further would push the command
   further into the limit.

When both hold, freeze the integrator. When either fails, resume.

The second condition is the one that is easy to drop and impossible to do without. Freezing on
saturation alone is a trap: the moment the road flattens, the loop needs the integrator to come
*down*, and a controller that is still saturated but now has negative error must be allowed to
integrate — otherwise the fix has produced a term that can never be reduced. Same-sign is what
makes the freeze one-directional: it blocks the accumulation that is going nowhere, and passes
the accumulation that is undoing it.

Clamping is popular because of what it does not require. There is no extra gain to choose and
no dynamics added; it is a switch on an existing path, and in a diagram it is one comparison and
an `if`. This is the method the source video teaches, and it is what the lecture demonstrates.

### Back-calculation, which is what `LimPID` already does

The Dyad block behind every controller in this lecture, `BlockComponents.Continuous.LimPID`,
ships an anti-windup scheme of its own — and it is not clamping. It is **back-calculation**, and
the difference is worth a minute in front of the room because the two produce visibly different
recoveries.

Instead of freezing the integrator, back-calculation *drives it back down*. It takes the
difference between what was commanded and what the actuator could deliver — the saturation
error, which is zero exactly when the actuator is listening — and feeds it
back into the integrator as a corrective input through a gain of `1/(k·Ni)`. The product
`Ni·Ti` is the time constant of that correction — it sets how fast the integrator is dragged
back toward a value consistent with what the actuator actually did. `Ni` defaults to 0.9 in
every loop in this lecture, and it is a number that has to be chosen.

The distinction in one line each:

- **Clamping** stops the integrator from getting worse. Nothing pulls it back, so on release it
  starts unwinding from wherever the freeze caught it.
- **Back-calculation** actively pulls the integrator toward the value that would have produced
  the delivered torque. By the time the actuator comes out of saturation, the integrator is
  already close to consistent with it.

Neither is uniformly better, and the comparison is worth making rather than asserting.
Back-calculation usually recovers with less overshoot, because it does not wait for release to
start correcting; clamping has nothing to tune and cannot be detuned wrong. `Ni` set badly gives
back-calculation its own failure modes — too short a tracking time constant and the correction
fights the controller, too long and it barely acts before the actuator is free again.

Three controllers, then: none, clamping, back-calculation. Same car, same hill.

> **Cell not written yet — the three-way comparison.**
>
> Blocked on the clamping controller variant, which does not exist yet — `LimPID` implements
> back-calculation and has no clamping mode. The cell needs speed and integrator state for all
> three on shared axes, and the peak overshoot of each. Nothing is stubbed here on purpose: a
> plot of invented data would look exactly like a result.

## 5. Clamp below the limit, not at it

One practical note to close on, because it is the part that is got wrong in real systems by
people who have implemented anti-windup correctly.

Clamping at exactly the nameplate limit still winds up.

An engine does not deliver 150 N.m. It delivers 150 N.m on a test bed, at sea level, at
operating temperature, when it was new. Hot, worn, at altitude, on poor fuel, it delivers less —
and the controller's saturation detector, which is comparing its own command against 150, sees
no saturation at all while the engine is pinned at 138. Every condition of section 1 is back:
the error will not close, the integrator does not know why, and it accumulates.

The same argument applies to every actuator with a datasheet. Valves stick, rudders meet
current, amplifiers sag under load. In each case the limit the controller believes and the limit
the hardware enforces are different numbers, and anti-windup keyed to the first one does nothing
about the second.

So set the clamp below the physical limit, with enough margin to cover the worst case the
actuator is allowed to degrade to. The cost is authority the controller declines to use. The
alternative is an anti-windup scheme that is correct in its logic and inert in practice, which
is the more expensive of the two — a fault that passes every bench test and appears in the
field.

## What this bought us

The loop now survives running out of actuator. That is a smaller claim than it sounds: nothing
here made the car able to climb the hill, and nothing made the speed drop go away. What went
away is the *second* failure — the one that arrives after the disturbance is over, that is
caused by the controller rather than the road, and that gets worse the longer the road stayed
steep.

Three things worth carrying out of this notebook.

- **Saturation is a modelling fact, not a tuning error.** It belongs in the model from the
  start, and the linear design is a design for a plant that does not exist.
- **The integrator's rule has an unstated assumption.** *Accumulate until the error is zero*
  assumes that accumulating more does something. Anti-windup, in any of its forms, is the
  business of detecting when it does not.
- **The command and the delivery are two different signals.** Every notebook before this one
  could plot one and mean the other. From here on they have to be plotted together.

**Notebook 07** goes after the derivative term the same way, with the same shape of argument: a
term whose rule is impeccable on an ideal signal, attached to a measurement that is not one.